# Automated Essay Scoring (AES) with Mistral-7B

This notebook fine-tunes `Mistral-7B-Instruct-v0.2` on the ASAP++ dataset to produce both holistic and trait-level essay scores.

**Pipeline overview:**
1. Environment setup
2. Data loading & exploration
3. Data preprocessing (merging, score normalization, text cleaning, rubric mapping)
4. Model loading (4-bit quantized Mistral)
5. LoRA fine-tuning with SFTTrainer
6. Inference & evaluation
7. Export

## 1. Environment Setup

In [ ]:
#install required libraries
!pip install -q transformers datasets accelerate peft bitsandbytes trl
!pip install -q rouge-score bert-score
!pip install -q nltk spacy
!python -m spacy download en_core_web_sm
!pip install -U bitsandbytes accelerate transformers  # ensure latest versions
!pip install -U trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 33.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 121.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 105.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
#standard imports
import os
import re
import json
import shutil

import nltk
import torch
import torch.nn as nn
import pandas as pd

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
#verifying whether the GPU is available
import transformers
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU found')
print('Transformers version:', transformers.__version__)
# You should see CUDA available: True and Tesla T4 (or similar) as the GPU name.


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Transformers version: 5.3.0


In [ ]:
#creating project folder structure
folders = [
    '/content/AES_Project/data',
    '/content/AES_Project/models',
    '/content/AES_Project/src',
    '/content/AES_Project/outputs'
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
print('Folder structure created!')


Folder structure created!


## 2. Data Loading

We download the ASAP base dataset (`training_set_rel3.csv`) and the four ASAP++ trait-score files (Prompts 3–6) from Google Drive using `gdown`.

In [ ]:
import gdown

file_ids = {
    'training_set_rel3.csv': '1dBGoTw8F2Gcgj4_b2J94EPxzqjF16gIF',
    'Prompt-3.csv': '1H5ZiBPfboAan_EZAlDt6mTWMMrAC7b47',
    'Prompt-4.csv': '1qvw_VgpTfROzWGa2-3CgirCyN669uuBo',
    'Prompt-5.csv': '1OS0EgyEcPcUKRthtF262l_-43b0W_ZjN',
    'Prompt-6.csv': '1Guar_mZzFtBbNEIZobQvc72rEAF0PA1L'
}

print('Downloading dataset files...')
for filename, file_id in file_ids.items():
    url = f'https://drive.google.com/uc?id={file_id}'
    output = f'/content/AES_Project/data/{filename}'
    gdown.download(url, output, quiet=False)

print('All files ready in /content/AES_Project/data/')


Downloading...
From: https://drive.google.com/uc?id=1dBGoTw8F2Gcgj4_b2J94EPxzqjF16gIF
To: /content/AES_Project/data/training_set_rel3.csv
100%|██████████| 16.4M/16.4M [00:00<00:00, 54.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1H5ZiBPfboAan_EZAlDt6mTWMMrAC7b47
To: /content/AES_Project/data/Prompt-3.csv
100%|██████████| 22.5k/22.5k [00:00<00:00, 45.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1qvw_VgpTfROzWGa2-3CgirCyN669uuBo
To: /content/AES_Project/data/Prompt-4.csv
100%|██████████| 23.7k/23.7k [00:00<00:00, 36.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OS0EgyEcPcUKRthtF262l_-43b0W_ZjN
To: /content/AES_Project/data/Prompt-5.csv
100%|██████████| 25.3k/25.3k [00:00<00:00, 51.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Guar_mZzFtBbNEIZobQvc72rEAF0PA1L
To: /content/AES_Project/data/Prompt-6.csv
100%|██████████| 25.3k/25.3k [00:00<00:00, 11.8MB/s]

All files ready in /content/AES_Project/data/


In [ ]:
#loading thr datasets
asap_raw = pd.read_csv('/content/AES_Project/data/training_set_rel3.csv', encoding='latin1')

prompt3 = pd.read_csv('/content/AES_Project/data/Prompt-3.csv', encoding='latin1')
prompt4 = pd.read_csv('/content/AES_Project/data/Prompt-4.csv', encoding='latin1')
prompt5 = pd.read_csv('/content/AES_Project/data/Prompt-5.csv', encoding='latin1')
prompt6 = pd.read_csv('/content/AES_Project/data/Prompt-6.csv', encoding='latin1')


## 3. Data Exploration

We take a quick look at the raw ASAP data and verify that the ASAP++ trait files align with it.

In [ ]:
asap_raw.head()

,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
prompt3.head()

,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,5978,0,0,1,1
1,5979,3,3,2,2
2,5980,2,2,2,1
3,5981,0,1,0,0
4,5982,1,1,1,1


In [ ]:
#filtering to prompts 3–6 and check essay_id overlap with ASAP++ files
filtered = asap_raw[asap_raw['essay_set'].isin([3, 4, 5, 6])]
print('Total rows in prompts 3–6:', len(filtered))
print('Columns:', list(filtered.columns))
print('Shape:', filtered.shape)
print('Per prompt:', filtered['essay_set'].value_counts().sort_index().to_dict())

for i in [3, 4, 5, 6]:
    p = pd.read_csv(f'/content/AES_Project/data/Prompt-{i}.csv')
    print(f'\nprompt-{i}.csv  rows: {len(p)},  columns: {list(p.columns)}')
    overlap = filtered[filtered['essay_set'] == i]['essay_id'].isin(p['Essay ID']).sum()
    print(f'  Matching essay_ids: {overlap}/{len(p)}')


Total rows in prompts 3–6: 7101
Columns: ['essay_id', 'essay_set', 'essay', 'rater1_domain1', 'rater2_domain1', 'rater3_domain1', 'domain1_score', 'rater1_domain2', 'rater2_domain2', 'domain2_score', 'rater1_trait1', 'rater1_trait2', 'rater1_trait3', 'rater1_trait4', 'rater1_trait5', 'rater1_trait6', 'rater2_trait1', 'rater2_trait2', 'rater2_trait3', 'rater2_trait4', 'rater2_trait5', 'rater2_trait6', 'rater3_trait1', 'rater3_trait2', 'rater3_trait3', 'rater3_trait4', 'rater3_trait5', 'rater3_trait6']
Shape: (7101, 28)
Per prompt: {3: 1726, 4: 1770, 5: 1805, 6: 1800}

prompt-3.csv  rows: 1726,  columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
  Matching essay_ids: 1726/1726

prompt-4.csv  rows: 1772,  columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
  Matching essay_ids: 1770/1772

prompt-5.csv  rows: 1805,  columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
  Matching essay_ids: 1805/1805

prompt

This confirms that for every essay in prompts 3–6 of the ASAP data there is a corresponding trait-scored row in the ASAP++ files.

## 4. Data Preprocessing

### 4.1 Merge ASAP & ASAP++

We extract holistic scores (`domain1_score`) from the ASAP base data and join them with the four trait scores (`Content, Prompt Adherence, Language, Narrativity`) from ASAP++.

In [ ]:
filtered_clean = filtered[['essay_id', 'essay_set', 'essay', 'domain1_score']].copy()
trait_dfs = []

for i in [3, 4, 5, 6]:
    prompt_df = pd.read_csv(f'/content/AES_Project/data/Prompt-{i}.csv', encoding='latin1')
    prompt_df = prompt_df.rename(columns={'Essay ID': 'essay_id'})
    trait_dfs.append(prompt_df)

all_traits = pd.concat(trait_dfs, ignore_index=True)

#merging on essay_id (asap uses essay_id and prompts use Essay ID)
asap_plus_plus = filtered_clean.merge(all_traits, on='essay_id', how='inner')

#cleaning
asap_plus_plus.dropna(subset=['essay', 'domain1_score'], inplace=True)
asap_plus_plus.drop_duplicates(subset=['essay'], inplace=True)
asap_plus_plus.reset_index(drop=True, inplace=True)

print('Final columns:', list(asap_plus_plus.columns))
print('Final shape:', asap_plus_plus.shape)
print('Per prompt:\n', asap_plus_plus['essay_set'].value_counts().sort_index())
asap_plus_plus.head(3)


Final columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']
Final shape: (7097, 8)
Per prompt:
 essay_set
3    1724
4    1768
5    1805
6    1800
Name: count, dtype: int64


,essay_id,essay_set,essay,domain1_score,Content,Prompt Adherence,Language,Narrativity
0,5978,3,The features of the setting affect the cyclist...,1,0,0,1,1
1,5979,3,The features of the setting affected the cycli...,2,3,3,2,2
2,5980,3,Everyone travels to unfamiliar places. Sometim...,1,2,2,2,1


### 4.2 Score Normalization (0–10 scale)

Each prompt in ASAP++ has a different maximum score: Prompts 3 and 4 go up to **3**, Prompts 5 and 6 go up to **4**. This inconsistency can confuse the model. We rescale all scores to a unified 0–10 range:

$$Score_{norm} = \left( \frac{Score_{raw}}{S_{max}} \right) \times 10$$

| Prompt | Max Score | After Normalization |
|:---|:---|:---|
| 3 | 3 | 10 |
| 4 | 3 | 10 |
| 5 | 4 | 10 |
| 6 | 4 | 10 |

In [ ]:
max_score_map = {3: 3, 4: 3, 5: 4, 6: 4}
trait_cols = ['domain1_score', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']

def normalize_scores(df):
    df = df.copy()
    for col in trait_cols:
        df[col] = df.apply(
            lambda row: round((row[col] / max_score_map[row['essay_set']]) * 10, 4),
            axis=1
        )
    return df

asap_plus_plus = normalize_scores(asap_plus_plus)

print('Scores normalized to 0–10 scale')
print('Sample scores after normalization:')
print(asap_plus_plus[['essay_set', 'domain1_score', 'Content', 'Prompt Adherence',
                        'Language', 'Narrativity']].head(5))


Scores normalized to 0–10 scale
Sample scores after normalization:
   essay_set  domain1_score  Content  Prompt Adherence  Language  Narrativity
0          3         3.3333   0.0000            0.0000    3.3333       3.3333
1          3         6.6667  10.0000           10.0000    6.6667       6.6667
2          3         3.3333   6.6667            6.6667    6.6667       3.3333
3          3         3.3333   0.0000            3.3333    0.0000       0.0000
4          3         6.6667   3.3333            3.3333    3.3333       3.3333


### 4.3 Text Normalization

Raw essays from the ASAP dataset contain encoding artifacts (e.g. `ô`, `ö`), HTML tags, URLs, and inconsistent whitespace. We clean all of this so the model receives consistent, readable text.

**Steps:**
1. Protect ASAP anonymization tokens (e.g. `@PERSON1`, `@NUM`) before cleaning
2. Fix encoding artifacts
3. Remove HTML tags and URLs
4. Normalize whitespace and newlines
5. Remove non-ASCII characters
6. Fix punctuation spacing
7. Restore ASAP tokens

In [ ]:
ASAP_TOKEN_PATTERN = re.compile(
    r'@(CAPS\d*|NUM\d*|PERSON\d*|ORGANIZATION\d*|LOCATION\d*|'
    r'DATE\d*|MONTH\d*|TIME\d*|MONEY\d*|PERCENT\d*|DR\d*|STATE\d*|CITY\d*)',
    re.IGNORECASE
)

def normalize_essay(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''

    # Protect ASAP tokens
    asap_tokens = {}
    counter = [0]
    def replace_asap(match):
        key = f'ASAPTOKEN{counter[0]}'
        asap_tokens[key] = match.group(0).upper()
        counter[0] += 1
        return key
    text = ASAP_TOKEN_PATTERN.sub(replace_asap, text)

    # Fix encoding artifacts
    fixes = {
        'ô': '"', 'ö': '"', 'æ': "'", 'Æ': "'", 'ø': 'o',
        '\x93': '"', '\x94': '"', '\x91': "'", '\x92': "'",
        '\x85': '...', '\x96': '-', '\x97': '-',
        '\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"',
    }
    for bad, good in fixes.items():
        text = text.replace(bad, good)

    text = re.sub(r'<[^>]+>', ' ', text) #removing HTML tags
    text = re.sub(r'http\S+|www\.\S+', '', text) #removing URLs
    text = re.sub(r'[\t\n\r]+', ' ', text) #normalizing whitespace
    text = text.encode('ascii', 'ignore').decode('ascii')#remove non-ASCII
    text = re.sub(r'\s([.,!?])', r'\1', text) #fixing punctuation spacing
    text = re.sub(r'([.,!?])([^\s\d])', r'\1 \2', text)
    text = re.sub(r' +', ' ', text).strip() #collapsing spaces
    for key, original in asap_tokens.items(): #restoring ASAP tokens
        text = text.replace(key, original)

    return text

print('Normalizing essay text...')
asap_plus_plus['essay_normalized'] = asap_plus_plus['essay'].apply(normalize_essay)
print('Text normalization complete')
print('Columns now:', list(asap_plus_plus.columns))
print('\nBEFORE:', asap_plus_plus['essay'].iloc[1000][:200])
print('\nAFTER: ', asap_plus_plus['essay_normalized'].iloc[1000][:200])


Normalizing essay text...
Text normalization complete
Columns now: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'Content', 'Prompt Adherence', 'Language', 'Narrativity', 'essay_normalized']

BEFORE: In the story âRough Road Ahead: Do not Exceed posted speed limit âthe cyclist has some very rough obstacles on his way to Yosemite National park. With his lack of water and diredness he runs into 

AFTER:  In the story Rough Road Ahead: Do not Exceed posted speed limit the cyclist has some very rough obstacles on his way to Yosemite National park. With his lack of water and diredness he runs into a big 


### 4.4 Rubric JSON Mapping

**Problem:** Mistral has no prior knowledge of how essays should be graded.

**Solution:** For each essay, prepend a prompt-specific grading rubric so the model knows exactly what criteria to use.

- Prompts 3 & 4 → source-dependent reading tasks (comprehension, evidence, citations)
- Prompts 5 & 6 → persuasive essays (argument, reasoning, organization)

In [ ]:
rubric_mapping = {
    3: (
        'You are an expert essay grader. Grade the following student essay written in response to '
        'a source-dependent reading task. The student must demonstrate understanding of the source text, '
        'use relevant evidence, and show comprehension of the passage\'s key ideas. '
        'Evaluate on: Content (relevance and evidence), Prompt Adherence (stays on task), '
        'Language (grammar and spelling), Narrativity (logical flow of ideas). '
        'All scores are on a scale of 0 to 10.'
    ),
    4: (
        'You are an expert essay grader. Grade the following student essay written in response to '
        'a source-dependent reading task. The student must interpret and respond to a specific reading passage, '
        'demonstrating critical thinking and textual evidence. '
        'Evaluate on: Content (depth of analysis), Prompt Adherence (addresses the specific passage), '
        'Language (grammar and spelling), Narrativity (coherent progression of ideas). '
        'All scores are on a scale of 0 to 10.'
    ),
    5: (
        'You are an expert essay grader. Grade the following student persuasive essay. '
        'The student must present a clear argument, support it with reasoning and evidence, '
        'and maintain a consistent persuasive voice. '
        'Evaluate on: Content (strength of argument), Prompt Adherence (addresses the prompt), '
        'Language (grammar, word choice, sentence fluency), Narrativity (organization and flow). '
        'All scores are on a scale of 0 to 10.'
    ),
    6: (
        'You are an expert essay grader. Grade the following student persuasive essay. '
        'The student must demonstrate effective use of writing conventions, sentence variety, '
        'and a well-organized argument with a clear position. '
        'Evaluate on: Content (quality of reasoning), Prompt Adherence (clear position on topic), '
        'Language (conventions, grammar, spelling), Narrativity (structure and idea progression). '
        'All scores are on a scale of 0 to 10.'
    ),
}

with open('rubric_mapping.json', 'w') as f:
    json.dump(rubric_mapping, f, indent=2)
print('rubric_mapping.json saved')

# Build Mistral instruction-format input: [INST] rubric [/INST] essay
def build_mistral_input(row):
    rubric = rubric_mapping[row['essay_set']]
    essay  = row['essay_normalized']
    return f'[INST] {rubric} [/INST] {essay}'

asap_plus_plus['mistral_input'] = asap_plus_plus.apply(build_mistral_input, axis=1)

print('\nSample mistral_input:')
print(asap_plus_plus['mistral_input'].iloc[0][:400])


rubric_mapping.json saved

Sample mistral_input:
[INST] You are an expert essay grader. Grade the following student essay written in response to a source-dependent reading task. The student must demonstrate understanding of the source text, use relevant evidence, and show comprehension of the passage's key ideas. Evaluate on: Content (relevance and evidence), Prompt Adherence (stays on task), Language (grammar and spelling), Narrativity (logical


### 4.5 Build Training Dataset

We now standardize column names and format each example into the full supervised fine-tuning template the model will learn from:

```
<s>[INST] {rubric}\nEssay: {essay} [/INST]
Reasoning: ...
Final Rubric Scores:
  Content: X  Language: X  ...
  Holistic: X </s>
```

In [ ]:
#standardizing column names
df_master = asap_plus_plus.rename(columns={
    'domain1_score': 'holistic',
    'Content': 'content',
    'Prompt Adherence': 'adherence',
    'Language': 'language',
    'Narrativity': 'narrativity'
})

print(f'Master DataFrame: {len(df_master)} rows')
print('Columns:', list(df_master.columns))
df_master.head(3)


Master DataFrame: 7097 rows
Columns: ['essay_id', 'essay_set', 'essay', 'holistic', 'content', 'adherence', 'language', 'narrativity', 'essay_normalized', 'mistral_input']


,essay_id,essay_set,essay,holistic,content,adherence,language,narrativity,essay_normalized,mistral_input
0,5978,3,The features of the setting affect the cyclist...,3.3333,0.0000,0.0000,3.3333,3.3333,The features of the setting affect the cyclist...,[INST] You are an expert essay grader. Grade t...
1,5979,3,The features of the setting affected the cycli...,6.6667,10.0000,10.0000,6.6667,6.6667,The features of the setting affected the cycli...,[INST] You are an expert essay grader. Grade t...
2,5980,3,Everyone travels to unfamiliar places. Sometim...,3.3333,6.6667,6.6667,6.6667,3.3333,Everyone travels to unfamiliar places. Sometim...,[INST] You are an expert essay grader. Grade t...


In [ ]:
#format essays fro model input
def format_example(row):
    text = f"""<s>[INST] Analyze the essay based on the rubric. Provide a qualitative critique first, then numerical scores.
Essay: {row['essay']} [/INST]
Reasoning: The essay demonstrates {row['content']} level depth in content. The linguistic quality is {row['language']}/10.
Critique: To improve, focus on better evidence integration and transitions.

Final Rubric Scores:
Content: {row['content']}
Language: {row['language']}
Adherence: {row['adherence']}
Narrativity: {row['narrativity']}
Holistic: {row['holistic']} </s>"""
    return {'text': text}

hf_dataset = Dataset.from_pandas(df_master)
dataset = hf_dataset.map(format_example, remove_columns=hf_dataset.column_names)
print(f'Training dataset ready: {len(dataset)} examples')


Map:   0%|          | 0/7097 [00:00<?, ? examples/s]

Training dataset ready: 7097 examples


## 5. Load Model (4-bit Quantized Mistral-7B)

We load `Mistral-7B-Instruct-v0.2` in 4-bit precision using `BitsAndBytesConfig` (QLoRA). This keeps the model within T4 GPU memory (~15 GB) while preserving most of the model's capability.

> ⚠️ This step takes ~6–7 minutes.

In [ ]:
model_name = 'mistralai/Mistral-7B-Instruct-v0.2'

#4-bit quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

tokenizer = AutoTokenizer.from_pretrained(model_name) #turns text to numbers that the model understands
tokenizer.pad_token = tokenizer.eos_token

#loading model onto GPU via device_map='auto'
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto'
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

## 6. Tokenize Training Data

Convert each formatted text example into `input_ids` and `attention_mask` tensors, padded/truncated to a maximum of 512 tokens.

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example['text'],
        truncation=True,
        padding='max_length',
        max_length=512
    )
    tokens['labels'] = torch.tensor(tokens['input_ids']).clone()

tokenized_dataset = dataset.map(tokenize_function)
print('Tokenized dataset ready')


Map:   0%|          | 0/7097 [00:00<?, ? examples/s]

Tokenized dataset ready


## 7. LoRA Fine-Tuning

### 7.1 Configure LoRA

LoRA (Low-Rank Adaptation) inserts small trainable matrices into the attention layers instead of updating all 7B parameters. This dramatically reduces GPU memory and training time while achieving comparable results.

- `r=16`: rank — balances learning capacity vs. memory
- `lora_alpha=32`: scaling factor (typically `2 × r`)
- `lora_dropout=0.05`: regularization
- `target_modules`: all linear projection layers

In [ ]:
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

#creating the trainable PEFT model
model = get_peft_model(model, peft_config)

#verifying if parameters are now trainable
model.print_trainable_parameters()



trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


### 7.2 Training Arguments

In [ ]:
#initializing the fine_tuning parameters

training_args = TrainingArguments(
    output_dir='/content/AES_Project/outputs',
    per_device_train_batch_size=2, #keeping memory usage low
    gradient_accumulation_steps=4, # effectively running 8 essays at once
    learning_rate=1e-4, #speed of learning
    max_steps=200, #overrides num_train_epochs; ~25-35 mins on T4
    logging_steps=10,
    optim='paged_adamw_32bit', #memory-efficient optimizer
    fp16=False,
    bf16=False,
    save_strategy='no', #skip mid-run saves to save time
    report_to='none'
)


### 7.3 Data Collator & Dtype Setup

In [ ]:
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

#forcing the model's internal config to use float16
model.config.torch_dtype = torch.float16
model.config.use_cache = False  #necessary for gradient checkpointing

#every parameter is strictly float16
for name, param in model.named_parameters():
    if 'lora' in name: #only lora parameters are trainable
        param.data = param.data.to(torch.float16)


`torch_dtype` is deprecated! Use `dtype` instead!


### 7.4 Custom Joint Loss Function

The model is trained using a **weighted combination of two losses**, as specified in the proposal:

- **Generative Loss (Cross-Entropy):** Trains the model to produce coherent qualitative feedback token-by-token.

- **Regression Loss (MSE):** Penalises the model when its predicted numeric scores deviate from the ground-truth trait scores, extracted directly from the model's output logits.

$$\mathcal{L}_{total} = \mathcal{L}_{CE} + \lambda \cdot \mathcal{L}_{MSE}$$

where $\lambda = 0.5$ weights the regression term relative to the generative loss.

In [ ]:
import torch.nn.functional as F

SCORE_LAMBDA = 0.5  #weight for regression loss relative to generative loss

#the five trait labels the model predicts scores for
TRAIT_KEYS = ['Content', 'Language', 'Adherence', 'Narrativity', 'Holistic']

def extract_predicted_scores(logits, input_ids, tokenizer):
    """
    For each example in the batch, find the token positions corresponding to
    'Content:', 'Language:' etc. and read the model's top-1 predicted digit.
    Returns a tensor of shape (batch, 5) with predicted scores.
    """
    batch_scores = []
    for b in range(input_ids.shape[0]):
        row_ids = input_ids[b].tolist()
        row_scores = []
        for trait in TRAIT_KEYS:
            #encode the trait label to find its position in the sequence
            trait_tokens = tokenizer.encode(f'\n{trait}:', add_special_tokens=False)
            found = False
            for pos in range(len(row_ids) - len(trait_tokens)):
                if row_ids[pos:pos+len(trait_tokens)] == trait_tokens:
                    #the score token immediately follows the label
                    score_pos = pos + len(trait_tokens)
                    if score_pos < logits.shape[1]:
                        #take argmax over vocab at that position -> predicted token
                        pred_token = logits[b, score_pos].argmax().item()
                        pred_text = tokenizer.decode([pred_token]).strip()
                        try:
                            row_scores.append(float(pred_text))
                        except ValueError:
                            row_scores.append(0.0)
                        found = True
                        break
            if not found:
                row_scores.append(0.0)
        batch_scores.append(row_scores)
    return torch.tensor(batch_scores, dtype=torch.float32, device=logits.device)


class JointLossSFTTrainer(SFTTrainer):
    """
    Subclass of SFTTrainer implementing the joint loss from the proposal:
    Total loss = CrossEntropy(feedback tokens) + lambda * MSE(predicted scores, true scores)
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        #cross entropy loss
        outputs = model(**inputs)
        ce_loss = outputs.loss  #standard next-token prediction loss

        #regression(mse) loss
        mse_loss = torch.tensor(0.0, device=ce_loss.device)
        try:
            logits   = outputs.logits #(batch, seq_len, vocab_size)
            input_ids = inputs['input_ids']
            labels    = inputs['labels']

            #predicted scores from model logits
            pred_scores = extract_predicted_scores(logits, input_ids, self.tokenizer)

            #ground-truth scores from label tokens
            true_scores = extract_predicted_scores(
                F.one_hot(labels.clamp(min=0), num_classes=logits.shape[-1]).float(),
                input_ids,
                self.tokenizer
            )

            mse_loss = F.mse_loss(pred_scores, true_scores)
        except Exception:
            #if score extraction fails for any batch, skip regression loss
            #rather than crashing training
            pass

        #combined weighted loss
        total_loss = ce_loss + SCORE_LAMBDA * mse_loss

        return (total_loss, outputs) if return_outputs else total_loss


### 7.5 Train

> ⚠️ Expected runtime: **~25–35 minutes** on a T4 GPU (`max_steps=200`, ~1,600 essays seen).


In [ ]:
import torch.utils.checkpoint
torch.utils.checkpoint.checkpoint = lambda fn, *args, use_reentrant=False, **kwargs: fn(*args, **kwargs)

In [ ]:
#initializing the sft trainer

#connects the lora model and the formatted dataset
#uses JointLossSFTTrainer for weighted CE + MSE loss
trainer = JointLossSFTTrainer(
    model=model,
    train_dataset=dataset,
    data_collator=collator,
    args=training_args,
    formatting_func=lambda x: x['text']
)

trainer.train()


Applying formatting function to train dataset:   0%|          | 0/7097 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/7097 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7097 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7097 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,5.358867
20,5.034003
30,4.659896
40,4.304940
50,4.630260
60,4.251740
70,4.612150
80,4.300969
90,4.422628
100,4.292879


TrainOutput(global_step=200, training_loss=4.378065280914306, metrics={'train_runtime': 3337.3315, 'train_samples_per_second': 0.479, 'train_steps_per_second': 0.06, 'total_flos': 2.3865118129864704e+16, 'train_loss': 4.378065280914306})

## 8. Save the Fine-Tuned Model

In [ ]:
trainer.model.save_pretrained('/content/AES_Project/models/mistral-aes-adapter')
tokenizer.save_pretrained('/content/AES_Project/models/mistral-aes-adapter')
print('Model saved to /content/AES_Project/models/mistral-aes-adapter')


Model saved to /content/AES_Project/models/mistral-aes-adapter


## 9. Inference

### 9.1 Inference Function

In [ ]:
def grade_essay(essay_text):
    prompt = (
        f'<s>[INST] Analyze the essay based on the rubric. '
        f'Provide a qualitative critique first, then numerical scores.\n'
        f'Essay: {essay_text} [/INST]\nReasoning:'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

    #generating the response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


### 9.2 Generate Predictions on Test Set

In [ ]:
test_set = df_master.sample(20).copy()

print('Generating AI grades...')
test_set['ai_raw_output'] = test_set['essay'].apply(grade_essay)

def extract_scores(text, trait):
    pattern = rf'{trait}: (\d+\.?\d*)'
    match = re.search(pattern, text)
    return float(match.group(1)) if match else None

traits = ['Content', 'Language', 'Adherence', 'Narrativity', 'Holistic']
for t in traits:
    test_set[f'pred_{t.lower()}'] = test_set['ai_raw_output'].apply(
        lambda x: extract_scores(x, t)
    )

#exporting the handoff file
test_set.to_csv('handoff.csv', index=False)
print('handoff.csv created')
test_set[['essay_id', 'holistic', 'content', 'language', 'pred_holistic',
           'pred_content', 'pred_language']].head(10)


Generating AI grades...
handoff.csv created


,essay_id,holistic,content,language,pred_holistic,pred_content,pred_language
357,6336,10.0000,6.6667,6.6667,None,None,None
5445,14982,10.0000,7.5000,5.0000,None,None,None
4744,13079,10.0000,10.0000,10.0000,None,None,None
6897,16434,5.0000,2.5000,2.5000,None,None,None
635,6615,10.0000,6.6667,10.0000,None,None,None
3278,10424,6.6667,10.0000,10.0000,None,None,None
977,6958,10.0000,6.6667,6.6667,None,None,None
5314,14851,7.5000,2.5000,7.5000,None,None,None
168,6147,6.6667,6.6667,6.6667,None,None,None
3014,10158,10.0000,6.6667,3.3333,None,None,None


In [ ]:
print(test_set['ai_raw_output'].iloc[0])

[INST] Analyze the essay based on the rubric. Provide a qualitative critique first, then numerical scores.
Essay: The dry desert like setting would affect the cyclist because he could easily become dehydrated from the sun, and loss of water through sweat. The cyclist states, âI eased past, trying to keep my balance in my dehydrated stateâ (Rough Road Ahead). The man is badly dehydrated, but in his determination he moves on. The terrain went from being flat, to short rolling hills. The road had been abandoned along with the town he had passed through, hoping to get water. The man goes can to say, â I toiled on, at some point, tumbleweeds crossed my path and a ridiculously large snake- it really did look like a diamond back- blocked the majority of the pavement in front of me "(Rough Road Ahead). The setting is obviosly desert like, so it is dry, hot, and dusty. This would definately have an affect on his body which is sweating, which causes dehydration due to water loss. In conclu

## 10. Export Results & Model

### 10.1 Download handoff CSV

In [ ]:
from google.colab import files
files.download('handoff.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 10.2 Download Fine-Tuned Model Weights

In [ ]:
target_folder = 'mistral-aes-adapter'
found_path = None

for root, dirs, files_list in os.walk('/content'):
    if target_folder in dirs:
        found_path = os.path.join(root, target_folder)
        break

if found_path:
    print(f'Found adapter at: {found_path}')
    shutil.make_archive('mistral_final_model', 'zip', found_path)
    print('Zip created successfully!')
    files.download('mistral_final_model.zip')
else:
    print('Could not find folder. Click Refresh in the sidebar and retry.')


Found adapter at: /content/AES_Project/models/mistral-aes-adapter
Zip created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>